## El primer gran proyecto - ¡Profesionalmente tú!

### Y, uso de herramientas.

### Pero primero: presentando Pushover

Pushover es una ingeniosa herramienta para enviar notificaciones push a tu teléfono.

Es muy fácil de configurar e instalar.

Sólo tienes que visitar https://pushover.net/ y hacer clic en "Login or Signup" en la parte superior derecha para registrarte y crear tus claves API.

Una vez que te hayas registrado, en la pantalla de inicio, haz clic en "Create an Application/API Token", dale un nombre cualquiera (como Agents) y haz clic en Create Application.

Luego agrega 2 líneas a tu archivo `.env`:

PUSHOVER_USER=_pon la clave que está en la parte superior derecha de tu pantalla de inicio de Pushover y que probablemente comienza con una u_  
PUSHOVER_TOKEN=_pon la clave cuando haces clic en tu nueva aplicación llamada Agentes (o lo que sea) y probablemente empiece por a_.

Recuerda guardar tu archivo `.env`, y ejecutar `load_dotenv(override=True)` después de guardar, para establecer tus variables de entorno.

Por último, haz clic en "Add Phone, Tablet or Desktop" para instalar en tu teléfono.




In [1]:
# imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# The usual start

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Usuario Pushover encontrado y comienza con {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token encontrado y comienza con {pushover_token[0]}")
else:
    print("Pushover token not found")

Usuario Pushover encontrado y comienza con u
Pushover token encontrado y comienza con a


In [4]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
push("HEY!!")

Push: HEY!!


In [6]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Registro de intereses de {name} con correo electrónico {email} y notas {notes}")
    return {"recorded": "ok"}

In [7]:
def record_unknown_question(question):
    push(f"Cliente {question} preguntó que no podía responder")
    return {"recorded": "ok"}

In [8]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Utilice esta herramienta para dejar constancia de que un usuario está interesado en estar en contacto y ha facilitado una dirección de correo electrónico.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user"
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it"
            }
            ,
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [9]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Utilice siempre esta herramienta para anotar cualquier pregunta que no haya podido responder por desconocer la respuesta.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered"
            },
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [10]:
tools = [{"type": "function", "function": record_user_details_json},
        {"type": "function", "function": record_unknown_question_json}]

In [11]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Utilice esta herramienta para dejar constancia de que un usuario está interesado en estar en contacto y ha facilitado una dirección de correo electrónico.',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this user'},
     'name': {'type': 'string',
      'description': "The user's name, if they provided it"},
     'notes': {'type': 'string',
      'description': "Any additional information about the conversation that's worth recording to give context"}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Utilice siempre esta herramienta para anotar cualquier pregunta que no haya podido responder por desconocer la respuesta.',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'de

In [12]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [13]:
globals()["record_unknown_question"]("esta es una pregunta realmente difícil")

Push: Cliente esta es una pregunta realmente difícil preguntó que no podía responder


{'recorded': 'ok'}

In [14]:
# This is a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Ed Donner"

In [16]:
system_prompt = f"Estás actuando como {name}. Está respondiendo a preguntas en el sitio web de {name}, \
en particular preguntas relacionadas con la carrera, formación, habilidades y experiencia de {name}. \
Su responsabilidad es representar a {name} para las interacciones en el sitio web lo más fielmente posible. \
Se le facilitará un resumen de la trayectoria profesional de {name} y su perfil de LinkedIn, que podrá utilizar para responder a las preguntas. \
Sé profesional y simpático, como si estuvieras hablando con un cliente potencial o un futuro empleador que ha visitado el sitio web. \
Si no sabes la respuesta a alguna pregunta, utiliza la herramienta record_unknown_question para anotar la pregunta que no has podido responder, aunque sea sobre algo trivial o no relacionado con la carrera. \
Si el usuario entabla una conversación, intenta que se ponga en contacto contigo por correo electrónico; pídele su dirección y anótala con la herramienta record_user_details. "



system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [17]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # This is the call to the LLM - see that we pass in the tools json

        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # If the LLM wants to call a tool, we do that!
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [18]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">- En primer lugar, ¡despliégalo por ti mismo! Es una herramienta real, valiosa - el futuro currículum..<br/>
            - A continuación, mejora los recursos: añade un mejor contexto sobre ti. Si conoce la GAR, añada una base de conocimientos sobre usted.<br/>
            - Añada más herramientas. ¿Podría tener una base de datos SQL con preguntas y respuestas comunes que el LLM pudiera leer y escribir?<br/>
            - Traiga el Evaluador del último laboratorio y añada otros patrones Agentic.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your career alter-ego) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>